# Rating Function Correlation Study

This notebook loads (or recomputes) the tournament simulation results and
computes pairwise correlations between rating functions, both for a single
representative context and averaged across all contexts. Results are shown as
text tables and seaborn heatmaps.


In [ ]:
import sys
import json
from pathlib import Path
from random import Random
from collections import defaultdict
from dataclasses import dataclass
from typing import Callable, Sequence

import matplotlib.pyplot as plt

sys.path.insert(0, str(
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
))

from tournament.generators import make_players, skilled_match
from tournament.engine import run_tournament
from tournament.pairing import make_record_group_pairing
from tournament.standings import compute_records, make_rank_key
from tournament.metrics import (
    standings_skill_correlation,
    mean_skill_gap,
    weighted_skill_gap_score,
    rematch_count,
)
from tournament.rating import (
    agent_differential,
    agent_ratio,
    agent_total_ratio,
    total_agents_scored,
    weighted_agent_differential,
    weighted_agent_total_ratio,
    linear_weights,
    exponential_weights,
)
from tournament.correlation import (
    correlation_matrix,
    display_correlation_matrix,
    plot_correlation_heatmap,
)

print('Imports loaded.')


## Setup: Rating functions, player pool, and simulation context


In [ ]:
# ── Rating functions ──────────────────────────────────────────────────────
_PRIMARY = [
    ("agent_differential",  agent_differential),
    ("agent_total_ratio",   agent_total_ratio),
    ("agent_ratio",         agent_ratio),
    ("total_agents_scored", total_agents_scored),
]

_DROP_OFF = [
    (f"{label} (last {k})", lambda seq, fn=fn, k=k: fn(seq[-k:]))
    for k in (2, 3, 4)
    for label, fn in _PRIMARY
]

_WEIGHTED = [
    (
        "weighted_diff (linear)",
        lambda seq: weighted_agent_differential(seq, linear_weights(len(seq))),
    ),
    (
        "weighted_diff (exp 1.2)",
        lambda seq: weighted_agent_differential(seq, exponential_weights(len(seq), base=1.2)),
    ),
    (
        "weighted_ratio (linear)",
        lambda seq: weighted_agent_total_ratio(seq, linear_weights(len(seq))),
    ),
]

RATING_FNS = _PRIMARY + _DROP_OFF + _WEIGHTED
print(f"{len(RATING_FNS)} rating functions registered.")

# ── Simulation parameters ──────────────────────────────────────────────────
N_PLAYERS = [i*2 for i in range(4, 21)]
N_ROUNDS  = range(3, 9)
BIG_SEEDS = [42, 123, 456, 789, 1011, 1213, 1415, 1617, 1819, 2021] + [i for i in range(2023, 2043)]
STRATEGY  = "adjacent"
TOUR_CTX  = [(i, j, seed) for i in N_PLAYERS for j in N_ROUNDS for seed in BIG_SEEDS]
print(f"{len(TOUR_CTX)} tournament contexts ({len(N_PLAYERS)} player counts × {len(list(N_ROUNDS))} round counts × {len(BIG_SEEDS)} seeds).")


In [ ]:
players = {(i, seed): make_players(i, Random(seed)) for i in N_PLAYERS for seed in BIG_SEEDS}
print(f"Generated {len(players)} player pools.")


## Load or run simulations

If a cached `big_results.pkl` exists in `../cache/`, it is loaded directly.
Otherwise the full simulation runs and the results are cached for next time.


In [ ]:
from tournament.models import Tournament


@dataclass(frozen=True)
class Result:
    """One tournament run under a single rating function."""
    label: str
    rating_fn: Callable
    tour: object
    corr: float
    gap: float
    weighted_gap: float
    rematches: int
    seed: int

    def to_dict(self) -> dict:
        return {
            "label": self.label,
            "tour": self.tour.to_dict(),
            "corr": self.corr,
            "gap": self.gap,
            "weighted_gap": self.weighted_gap,
            "rematches": self.rematches,
            "seed": self.seed,
        }

    @classmethod
    def from_dict(cls, d: dict, rating_fn: Callable) -> "Result":
        return cls(
            label=d["label"],
            rating_fn=rating_fn,
            tour=Tournament.from_dict(d["tour"]),
            corr=d["corr"],
            gap=d["gap"],
            weighted_gap=d["weighted_gap"],
            rematches=d["rematches"],
            seed=d["seed"],
        )


def run_comparison(n_players: int, n_rounds: int, seed: int) -> list[Result]:
    results = []
    for label, fn in RATING_FNS:
        rng = Random(seed)
        tour = run_tournament(
            players[(n_players, seed)],
            n_rounds=n_rounds,
            pairing=make_record_group_pairing(STRATEGY, rating_fn=fn),
            rng=rng,
            match_model=skilled_match,
        )
        results.append(Result(
            label=label,
            rating_fn=fn,
            tour=tour,
            corr=standings_skill_correlation(tour),
            gap=mean_skill_gap(tour),
            weighted_gap=weighted_skill_gap_score(tour),
            rematches=rematch_count(tour),
            seed=seed,
        ))
    return results


def run_mass_comparisons(contexts):
    return {ctx: run_comparison(*ctx) for ctx in contexts}


In [ ]:
CACHE_DIR  = Path("../cache")
CACHE_FILE = CACHE_DIR / "big_results.json"


def _results_to_json(results_by_ctx) -> dict:
    """Serialize big_results to a JSON-compatible dict."""
    return {
        f"{ctx[0]},{ctx[1]},{ctx[2]}": [r.to_dict() for r in results]
        for ctx, results in results_by_ctx.items()
    }


def _results_from_json(data: dict) -> dict:
    """Rebuild big_results from a JSON-loaded dict."""
    fn_by_label = dict(RATING_FNS)
    out = {}
    for key, result_dicts in data.items():
        ctx = tuple(int(x) for x in key.split(","))
        out[ctx] = [Result.from_dict(d, fn_by_label[d["label"]]) for d in result_dicts]
    return out


def load_or_run(contexts, cache_file=CACHE_FILE):
    """Load cached results if available, otherwise run and cache."""
    if cache_file.exists():
        with cache_file.open("r") as f:
            cached = _results_from_json(json.load(f))
        cached_ctxs = set(cached.keys())
        needed = set(contexts)
        if needed.issubset(cached_ctxs):
            print(f"Loaded {len(needed)} contexts from {cache_file}")
            return {ctx: cached[ctx] for ctx in contexts}
        missing = needed - cached_ctxs
        print(f"Cache has {len(cached_ctxs)} contexts; running {len(missing)} missing...")
        new_results = run_mass_comparisons(missing)
        cached.update(new_results)
        with cache_file.open("w") as f:
            json.dump(_results_to_json(cached), f)
        return {ctx: cached[ctx] for ctx in contexts}
    print(f"No cache at {cache_file}; running full simulation...")
    results = run_mass_comparisons(contexts)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    with cache_file.open("w") as f:
        json.dump(_results_to_json(results), f)
    print(f"Cached {len(results)} contexts to {cache_file}")
    return results


big_results = load_or_run(TOUR_CTX)
print(f"big_results: {len(big_results)} contexts, {len(next(iter(big_results.values())))} rating functions each.")


## Single-context correlation

Pick one representative tournament (32 players, 7 rounds, seed 42) and compute
the full pairwise correlation matrix across all rating functions.


In [ ]:
demo_ctx = (32, 7, 42)
demo_results = big_results[demo_ctx]
labels = [res.label for res in demo_results]

# ── Rank correlation (Spearman-style) ──────────────────────────────────────
rank_matrix = correlation_matrix(demo_results, mode="rank")
print(display_correlation_matrix(rank_matrix, labels, f"Rank correlation — {demo_ctx}"))
fig1 = plot_correlation_heatmap(rank_matrix, labels, f"Rank correlation — {demo_ctx}", mode="rank")
plt.show()


In [ ]:
# ── Score correlation (Pearson on raw rating scores) ───────────────────────
score_matrix = correlation_matrix(demo_results, mode="score")
print(display_correlation_matrix(score_matrix, labels, f"Score correlation — {demo_ctx}"))
fig2 = plot_correlation_heatmap(score_matrix, labels, f"Score correlation — {demo_ctx}", mode="score")
plt.show()


## Averaged correlation across all contexts

Compute the correlation matrix for every context in `big_results`, then
average pairwise to get a single robust matrix.


In [ ]:
def average_correlation_matrices(results_by_ctx, mode="rank"):
    """Compute correlation_matrix for every context and average pairwise."""
    sums   = defaultdict(float)
    counts = defaultdict(int)
    for ctx, ctx_results in results_by_ctx.items():
        mat = correlation_matrix(ctx_results, mode=mode)
        for pair, val in mat.items():
            sums[pair] += val
            counts[pair] += 1
    return {pair: sums[pair] / counts[pair] for pair in sums}

avg_rank = average_correlation_matrices(big_results, mode="rank")
print(display_correlation_matrix(avg_rank, labels, "Averaged rank correlation — all contexts"))
fig3 = plot_correlation_heatmap(avg_rank, labels, "Averaged rank correlation — all contexts", mode="rank")
plt.show()


In [ ]:
avg_score = average_correlation_matrices(big_results, mode="score")
print(display_correlation_matrix(avg_score, labels, "Averaged score correlation — all contexts"))
fig4 = plot_correlation_heatmap(avg_score, labels, "Averaged score correlation — all contexts", mode="score")
plt.show()


## Interpretation

- **Rank correlation ≈ 1**: the two rating functions produce nearly identical
  standings orderings. They are redundant as tiebreakers.
- **Rank correlation ≈ 0**: the functions disagree on how to rank players within
  record groups. They capture different information.
- **Score correlation vs. rank correlation**: if score correlation is much lower
  than rank correlation, the functions agree on *order* but disagree on *how far
  apart* players are. This matters if the score magnitude feeds into pairing
  decisions.
- **Drop-off variants** (last k rounds) should correlate highly with their
  parent function when k ≥ n_rounds, and diverge as k decreases.
- **Averaged vs. single context**: if the averaged matrix looks similar to the
  single-context one, the correlations are stable across tournament sizes. If
  not, some functions may be more/less redundant depending on player count or
  round count.
